In [1]:
# Need to run from base environment for grib, which is miniconda3/bin/python
# to run code in terminal using that environment, run this command: /home/csutter/miniconda3/bin/python /home/csutter/DRIVE-clean/weather_events/notebooks/cam_modelpred_QPE_active.py <-- update this last part w the script you want to run

# This code just investigates MRMS data. Main code to run for prepping operational files is /home/csutter/DRIVE-clean/weather_events/notebooks/cam_modelpred_QPE.py

import xarray as xr
import requests
import gzip
import os
import pandas as pd
import numpy as np


In [2]:
import sys
import xarray
print(sys.executable)
print(xarray.__file__)

/home/csutter/miniconda3/bin/python
/home/csutter/miniconda3/lib/python3.9/site-packages/xarray/__init__.py


Data documentation: https://www.nssl.noaa.gov/projects/mrms/operational/tables.php

Data source: https://noaa-mrms-pds.s3.amazonaws.com/index.html#CONUS/PrecipFlag_00.00/

For precip type (which is also in MRMS data, just different GRIB file)

In [22]:
import os
import gzip
import glob
import requests
import xarray as xr
import pandas as pd
import numpy as np

def get_precipflag_s3(ts):
    """
    ts: pandas Timestamp (UTC)
    """
    date_str = ts.strftime('%Y%m%d')
    time_str = ts.strftime('%H%M') + "00"
    
    # 1. Updated for PrecipFlag
    base_url = "https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipFlag_00.00"
    file_name = f"MRMS_PrecipFlag_00.00_{date_str}-{time_str}.grib2.gz"
    full_url = f"{base_url}/{date_str}/{file_name}"
    
    print(f"Attempting S3 Download: {full_url}")
    
    # 2. Download
    r = requests.get(full_url)

    if r.status_code == 200:
        temp_grib = f"temp_{ts.strftime('%Y%m%d%H%M%S')}.grib2" 
        with open(temp_grib, "wb") as f:
            f.write(gzip.decompress(r.content))

        # 3. Open and IMMEDIATELY subset to the NY region
        ds_full = xr.open_dataset(temp_grib, engine="cfgrib", backend_kwargs={'indexpath': ''})
        ds = ds_full.sel(latitude=slice(47.5, 38.5), longitude=slice(278.0, 291.0)).load()
        
        # CLOSE the file handle
        ds_full.close()
        del ds_full
        
        # 4. === THE "GLANCE" LOGIC ===
        # cfgrib often names custom MRMS parameters 'unknown'
        # We dynamically grab the first data variable name in the dataset
        var_name = list(ds.data_vars)[0]
        print(f"\n--- Data Glance for {time_str} ---")
        print(f"Variable name in xarray: '{var_name}'")
        
        # Flatten the array, drop NaNs, and count unique integer flags
        flat_data = ds[var_name].values.flatten()
        valid_data = flat_data[~np.isnan(flat_data)]
        unique_vals, counts = np.unique(valid_data, return_counts=True)
        
        flag_dict = {
                    -3: "No Coverage",
                    0: "No Precip", 
                    1: "Warm Rain", 
                    3: "Snow", 
                    6: "Convective Rain", 
                    7: "Hail/Mixed", 
                    10: "Cold Rain",
                    91: "Tropical Stratiform",
                    96: "Tropical Convective"
                }
                     
        for val, count in zip(unique_vals, counts):
            name = flag_dict.get(int(val), "Other/Missing")
            print(f"Flag {int(val)} ({name}): {count} pixels")
        print("----------------------------------\n")
        
        # 5. Cleanup
        if os.path.exists(temp_grib):
            os.remove(temp_grib)
            for f in glob.glob(f"{temp_grib}*.idx"):
                os.remove(f)
        
        return ds

    else:
        print(f"Failed. Status Code: {r.status_code}")
        if r.status_code in [403, 404]:
            print("Check: Ensure the time ends in a multiple of 2 (e.g., 02, 04, 10).")
        return None

# Execute
time_str = "20260125_1700"  
raw_ts = pd.to_datetime(time_str, format="%Y%m%d_%H%M")
even_min = (raw_ts.minute // 2) * 2
mrms_ts = raw_ts.replace(minute=even_min, second=0)

flag_ds = get_precipflag_s3(mrms_ts)

Attempting S3 Download: https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipFlag_00.00/20260125/MRMS_PrecipFlag_00.00_20260125-170000.grib2.gz

--- Data Glance for 170000 ---
Variable name in xarray: 'unknown'
Flag 0 (No Precip): 417849 pixels
Flag 3 (Snow): 726969 pixels
Flag 10 (Cold Rain): 25182 pixels
----------------------------------



For QPE

In [11]:

def get_qpe_s3(ts):
    """
    ts: pandas Timestamp (UTC)
    """
    # 1. Format the S3 URL for NOAA Open Data
    # Path: https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipRate_00.00/YYYYMMDD/MRMS_PrecipRate_00.00_YYYYMMDD-HHMMSS.grib2.gz
    date_str = ts.strftime('%Y%m%d')
    # MRMS files are every 2 mins. Let's force it to 00 minutes for the test.
    time_str = ts.strftime('%H%M') + "00"
    
    # For QPE
    # base_url = "https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipRate_00.00"
    # file_name = f"MRMS_PrecipRate_00.00_{date_str}-{time_str}.grib2.gz"
    # full_url = f"{base_url}/{date_str}/{file_name}"

    # For Precip flag
    base_url = "https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipFlag_00.00"
    file_name = f"MRMS_PrecipFlag_00.00_{date_str}-{time_str}.grib2.gz"
    full_url = f"{base_url}/{date_str}/{file_name}"

    
    print(f"Attempting S3 Download: {full_url}")
    
    # 2. Download
    r = requests.get(full_url)

    # if r.status_code == 200:
    #     temp_grib = "test_qpe.grib2"
    #     with open(temp_grib, "wb") as f:
    #         # We decompress the .gz before saving
    #         f.write(gzip.decompress(r.content))

    #  2. download (updating to before parallelizing)
    if r.status_code == 200:
        # Use a unique name for each thread based on the timestamp string
        temp_grib = f"temp_{ts.strftime('%Y%m%d%H%M%S')}.grib2" 
        with open(temp_grib, "wb") as f:
            f.write(gzip.decompress(r.content))

        # 3. Open and IMMEDIATELY subset to NY box
        ds_full = xr.open_dataset(temp_grib, engine="cfgrib", backend_kwargs={'indexpath': ''})
        
        # New York roughly: Lat 40-45.5, Lon 280-290
        # Subsetting here shrinks the data in RAM by ~90%
        # ds = ds_full.sel(latitude=slice(46, 39), longitude=slice(279, 291)).load()
        # Expanded Box: Covers NY + NJ, PA, CT, and Southern Canada
        ds = ds_full.sel(latitude=slice(47.5, 38.5), longitude=slice(278.0, 291.0)).load()
        
        # CLOSE the file handle (Critical!)
        ds_full.close()
        del ds_full
        
        # NOW delete the physical files from the hard drive
        if os.path.exists(temp_grib):
            os.remove(temp_grib)
            
            # cfgrib creates index files that look like 'temp_123.grib2.923a8.idx'
            # We can use a wildcard to catch any .idx file associated with this temp file
            import glob
            for f in glob.glob(f"{temp_grib}*.idx"):
                os.remove(f)
        
        return ds

    else:
        print(f"Failed. Status Code: {r.status_code}")
        if r.status_code == 403 or r.status_code == 404:
            print("Check: Ensure the time ends in a multiple of 2 (e.g., 02, 04, 10).")
        return None

In [12]:
# Execute 

# 1. SETUP PARAMETERS
time_str = "20250412_0000"  # Your input format
THRESHOLD = 0.1             # mm/hr (The "Active" cutoff)

# 2. TIME ALIGNMENT (The 2-Minute Even Rule)
# Convert string to Timestamp
raw_ts = pd.to_datetime(time_str, format="%Y%m%d_%H%M")

# MRMS files are every 2 mins (usually even). Round down to be safe.
even_min = (raw_ts.minute // 2) * 2
mrms_ts = raw_ts.replace(minute=even_min, second=0)

print(f"Original Time: {raw_ts}")
print(f"Targeting MRMS File: {mrms_ts.strftime('%Y%m%d-%H%M00')}")

# 3. GET DATA (Using your existing function)
qpe_ds = get_qpe_s3(mrms_ts)

Original Time: 2025-04-12 00:00:00
Targeting MRMS File: 20250412-000000
Attempting S3 Download: https://noaa-mrms-pds.s3.amazonaws.com/CONUS/PrecipFlag_00.00/20250412/MRMS_PrecipFlag_00.00_20250412-000000.grib2.gz


In [13]:
qpe_ds

<xarray.Dataset>
Dimensions:         (latitude: 900, longitude: 1300)
Coordinates:
    time            datetime64[ns] 2025-04-12
    step            timedelta64[ns] 00:00:00
    heightAboveSea  float64 0.0
  * latitude        (latitude) float64 47.5 47.49 47.48 ... 38.53 38.52 38.51
  * longitude       (longitude) float64 278.0 278.0 278.0 ... 291.0 291.0 291.0
    valid_time      datetime64[ns] 2025-04-12
Data variables:
    unknown         (latitude, longitude) float32 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  161
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             161
    history:                 2026-03-13T14:51 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:
# FOR QPE

# 1. Rename 'unknown' to 'precip_rate' for clarity
ds = qpe_ds.rename({'unknown': 'precip_rate'})

# 2. Slice to a New York bounding box
# Lat: 40 to 45, Lon: 280 to 290 (roughly -80 to -70 W)
ny_slice = ds.sel(latitude=slice(45, 40), longitude=slice(280, 290))

# 3. Check the max value
max_precip = ny_slice.precip_rate.max().values
print(f"Max Precip Rate in NY slice: {max_precip} mm/hr")

# 4. Check for 'Missing' values
# MRMS uses -999 for missing/no coverage. 
# We should mask those out.

ny_clean = ny_slice.where(ny_slice.precip_rate >= 0)

print(f"Average precip where it was actually snowing/raining: {ny_clean.precip_rate.mean().values} mm/hr")

Max Precip Rate in NY slice: 129.60000610351562 mm/hr
Average precip where it was actually snowing/raining: 0.013453599065542221 mm/hr


In [14]:
# FOR Precip flag

# 1. Rename 'unknown' to 'precip_rate' for clarity
ds = qpe_ds.rename({'unknown': 'precip_flag'})

# 2. Slice to a New York bounding box
# Lat: 40 to 45, Lon: 280 to 290 (roughly -80 to -70 W)
ny_slice = ds.sel(latitude=slice(45, 40), longitude=slice(280, 290))

# 3. Check the max value
max_precip = ny_slice.precip_flag.max().values
print(f"Max Precip Rate in NY slice: {max_precip} mm/hr")

# 4. Check for 'Missing' values
# MRMS uses -999 for missing/no coverage. 
# We should mask those out.

ny_clean = ny_slice.where(ny_slice.precip_flag >= 0)

print(f"Average precip where it was actually snowing/raining: {ny_clean.precip_flag.mean().values} mm/hr")

Max Precip Rate in NY slice: 10.0 mm/hr
Average precip where it was actually snowing/raining: 1.1392539739608765 mm/hr


In [16]:
print(type(ny_clean))

<class 'xarray.core.dataset.Dataset'>


In [15]:
ny_clean

<xarray.Dataset>
Dimensions:         (latitude: 500, longitude: 1000)
Coordinates:
    time            datetime64[ns] 2025-04-12
    step            timedelta64[ns] 00:00:00
    heightAboveSea  float64 0.0
  * latitude        (latitude) float64 45.0 44.99 44.98 ... 40.03 40.02 40.01
  * longitude       (longitude) float64 280.0 280.0 280.0 ... 290.0 290.0 290.0
    valid_time      datetime64[ns] 2025-04-12
Data variables:
    precip_flag     (latitude, longitude) float32 0.0 0.0 0.0 ... 1.0 1.0 1.0
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  161
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             161
    history:                 2026-03-13T14:51 GRIB to CDM+CF via cfgrib-0.9.1...

# Notes
- LG paper (https://journals.ametsoc.org/view/journals/atsc/76/11/jas-d-19-0004.1.xml) uses AHPS for QPE data - National Weather Service Advanced Hydrologic Prediction Service (AHPS). 
    - But that data is daily (24h)
    - Have to do manual Z-R for LE snow vs rain (in LG paper)
    - [Gemini, need to check] AHPS: Usually a 24-hour total. It is "Gauge-Biased," meaning they take the radar and "force" it to match rain gauges.
- MRMS 
    - [Gemini] "MRMS: This is the modern successor. It provides the same radar+gauge blend but at 1-hour or even 2-minute intervals. Since your camera model is likely looking at 5- or 10-minute loops, MRMS is much better for your project than the 24-hour AHPS total mentioned in the paper.
    - Hourly! We want this version. 
- Data source: https://mesonet.agron.iastate.edu/rainfall/
    - I believe the first "About QPE estimates" is the AHPS data used by LG
    - The MRMS data blurb is below - I want to use this. 
- To consider: what QPE threshold to use to filter whether it's precipitating or not.
    - [Gemini] Is 0.1 mm/hr safe for Rain vs. Snow?The short answer is yes, but with a small caveat for snow.Rain ($0.1$ is very safe): Rain is quite "reflective." If the radar says $0.1$ mm/hr for rain, it is almost certainly raining.Snow ($0.1$ is a bit "noisy"): Snow is much less dense than rain and reflects less energy.The Risk: At very low thresholds ($< 0.1$), the radar might pick up "clear-air echoes" (dust, bugs, or even birds) that it accidentally labels as tiny amounts of snow.The Recommendation: For your "Active Storm" filter, 0.1 mm/hr is an excellent starting point for both. It is high enough to ignore most "noise" but low enough to catch "light rain" and "moderate flurries."Pro-Tip: In the weather world, we often use 0.25 mm/hr (roughly $0.01$ inches/hr) as the "standard" threshold for "measurable" precipitation. If you find $0.1$ gives you too many "Active" locations that look clear on camera, bump it up to $0.25$.